In [1]:
import os
import pandas as pd

# Đường dẫn file dữ liệu gốc
DATA_PATH = '../data/raw/steam_games.csv'

# Đọc dữ liệu
df = pd.read_csv(DATA_PATH)

# 1. Kiểm tra kích thước và trùng lặp khóa chính
total_rows, total_cols = df.shape
duplicate_ids = df['steam_appid'].duplicated().sum()

print("=" * 60)
print(f"1. Tổng số dòng thực tế: {total_rows:,} dòng")
print(f"2. Tổng số cột thực tế: {total_cols} cột")
print(f"3. Số mã game (steam_appid) bị trùng lặp: {duplicate_ids}")
print("=" * 60)

# 2. Lập bảng thống kê 21 cột (Kiểu dữ liệu, Số lượng thiếu, % Thiếu)
audit_summary = pd.DataFrame({
    'Tên cột': df.columns,
    'Kiểu dữ liệu': df.dtypes.values,
    'Số lượng Null': df.isnull().sum().values,
    'Tỷ lệ Null (%)': (df.isnull().sum() / len(df) * 100).round(2).values,
    'Ví dụ giá trị': [df[col].dropna().iloc[0] if df[col].notnull().any() else 'N/A' for col in df.columns]
})

display(audit_summary)

1. Tổng số dòng thực tế: 71,429 dòng
2. Tổng số cột thực tế: 21 cột
3. Số mã game (steam_appid) bị trùng lặp: 0


,Tên cột,Kiểu dữ liệu,Số lượng Null,Tỷ lệ Null (%),Ví dụ giá trị
0,steam_appid,int64,0,0.0,2719580
1,name,object,0,0.0,勇者の伝説の勇者
2,developers,object,0,0.0,['ぽけそう']
3,publishers,object,0,0.0,['ぽけそう']
4,categories,object,0,0.0,"['Single-player', 'Family Sharing']"
5,genres,object,0,0.0,"['Casual', 'Indie']"
6,required_age,int64,0,0.0,0
7,n_achievements,int64,0,0.0,0
8,platforms,object,0,0.0,['windows']
9,is_released,bool,0,0.0,True


In [2]:
# Kiểm tra sâu các chỉ số kinh doanh quan trọng
total_games = len(df)

# 1. Soi cột Reviews
zero_review_games = (df['total_reviews'] == 0).sum()
zero_review_pct = (zero_review_games / total_games) * 100

qualified_games_50 = (df['total_reviews'] >= 50).sum()
qualified_pct = (qualified_games_50 / total_games) * 100

print(f"--- KIỂM TRA ĐỘ LỆCH REVIEWS ---")
print(f"- Game có 0 review (chưa có tương tác): {zero_review_games:,} ({zero_review_pct:.2f}%)") 
print(f"- Game đạt chuẩn >= 50 reviews (đủ điều kiện xếp hạng): {qualified_games_50:,} ({qualified_pct:.2f}%)")
print(f"- Số review cao nhất của 1 game: {df['total_reviews'].max():,}")
print(f"- Trung vị review toàn sàn: {df['total_reviews'].median()} (Trung bình: {df['total_reviews'].mean():.1f})")

--- KIỂM TRA ĐỘ LỆCH REVIEWS ---
- Game có 0 review (chưa có tương tác): 26,812 (37.54%)
- Game đạt chuẩn >= 50 reviews (đủ điều kiện xếp hạng): 11,568 (16.20%)
- Số review cao nhất của 1 game: 1,279,700
- Trung vị review toàn sàn: 2.0 (Trung bình: 410.2)


In [3]:
# Kiểm tra cột giá tiền thực tế
price_col = 'price_initial (USD)'

print("--- KIỂM TRA PHÂN PHỐI GIÁ TIỀN ---")
print(f"Giá thấp nhất: ${df[price_col].min()}")
print(f"Giá cao nhất: ${df[price_col].max()}")
print(f"Trung vị giá: ${df[price_col].median()}")
print(f"Trung bình giá: ${df[price_col].mean():.2f}")

# Đếm số lượng game Free vs Trả phí
free_games = (df[price_col] == 0).sum()
paid_games = (df[price_col] > 0).sum()
print(f"\nSố game Free ($0): {free_games:,} ({free_games/len(df)*100:.2f}%)")
print(f"Số game Trả phí (> $0): {paid_games:,} ({paid_games/len(df)*100:.2f}%)")

# Top 5 mức giá phổ biến nhất trên Steam
print("\n--- TOP 10 MỨC GIÁ PHỔ BIẾN NHẤT ---")
print(df[price_col].value_counts().head(10))

--- KIỂM TRA PHÂN PHỐI GIÁ TIỀN ---
Giá thấp nhất: $0.0
Giá cao nhất: $999.98
Trung vị giá: $2.99
Trung bình giá: $6.53

Số game Free ($0): 22,434 (31.41%)
Số game Trả phí (> $0): 48,995 (68.59%)

--- TOP 10 MỨC GIÁ PHỔ BIẾN NHẤT ---
price_initial (USD)
0.00     22434
4.99      6752
9.99      5956
0.99      5618
1.99      4195
2.99      4000
14.99     2946
3.99      2838
19.99     2756
5.99      2107
Name: count, dtype: int64


In [4]:
# 1. Danh sách đầy đủ toàn bộ 6 cột lưu dạng mảng trong dataset
all_array_cols = ['genres', 'categories', 'platforms', 'developers', 'publishers', 'additional_content']

print("--- 1. MẪU DỮ LIỆU CỦA TOÀN BỘ 6 CỘT DẠNG MẢNG ---")
display(df[all_array_cols].head(3))

# 2. Kiểm tra tỷ lệ mảng rỗng '[]' trên từng cột để tìm dữ liệu thiếu ẩn
empty_counts = {}
for col in all_array_cols:
    empty_cnt = (df[col].astype(str).str.strip().isin(['[]', '', 'nan', 'None', 'NaN'])).sum()
    empty_counts[col] = {
        'Số game mảng rỗng []': empty_cnt,
        'Tỷ lệ rỗng (%)': round(empty_cnt / len(df) * 100, 2),
        'Số game có dữ liệu': len(df) - empty_cnt,
        'Tỷ lệ có dữ liệu (%)': round((len(df) - empty_cnt) / len(df) * 100, 2)
    }

empty_df = pd.DataFrame(empty_counts)
print("\n--- 2. KIỂM TOÁN TỶ LỆ RỖNG TRÊN TỪNG CỘT MẢNG ---")
display(empty_df)

# 3. Xem phân bố thực tế của nền tảng (OS)
print("\n--- 3. PHÂN BỐ NỀN TẢNG (PLATFORMS) PHỔ BIẾN NHẤT ---")
print(df['platforms'].value_counts().head(5))

--- 1. MẪU DỮ LIỆU CỦA TOÀN BỘ 6 CỘT DẠNG MẢNG ---


,genres,categories,platforms,developers,publishers,additional_content
0,"['Casual', 'Indie']","['Single-player', 'Family Sharing']",['windows'],['ぽけそう'],['ぽけそう'],[]
1,"['Action', 'Adventure']","['Single-player', 'Multi-player', 'Co-op', 'On...",['windows'],['Hello Games'],['Hello Games'],[]
2,"['RPG', 'Strategy']","['Single-player', 'Multi-player', 'PvP', 'Shar...","['windows', 'mac']",['GoldenGod Games'],['GoldenGod Games'],[]



--- 2. KIỂM TOÁN TỶ LỆ RỖNG TRÊN TỪNG CỘT MẢNG ---


,genres,categories,platforms,developers,publishers,additional_content
Số game mảng rỗng [],223.00,558.00,0.0,200.00,493.00,64266.00
Tỷ lệ rỗng (%),0.31,0.78,0.0,0.28,0.69,89.97
Số game có dữ liệu,71206.00,70871.00,71429.0,71229.00,70936.00,7163.00
Tỷ lệ có dữ liệu (%),99.69,99.22,100.0,99.72,99.31,10.03



--- 3. PHÂN BỐ NỀN TẢNG (PLATFORMS) PHỔ BIẾN NHẤT ---
platforms
['windows']                    54889
['windows', 'mac', 'linux']     7362
['windows', 'mac']              6539
['windows', 'linux']            2611
['mac']                           21
Name: count, dtype: int64


In [5]:
print("--- KIỂM TRA NGÀY PHÁT HÀNH (RELEASE_DATE) ---")

# 1. Kiểm tra cờ is_released
released_counts = df['is_released'].value_counts()
print(f"Số game đã phát hành chính thức: {released_counts.get(True, 0):,} ({released_counts.get(True, 0)/len(df)*100:.2f}%)")
print(f"Số game chưa phát hành / Sắp ra mắt: {released_counts.get(False, 0):,} ({released_counts.get(False, 0)/len(df)*100:.2f}%)")

# 2. Thử chuyển đổi sang datetime xem có bao nhiêu dòng bị lỗi định dạng
parsed_dates = pd.to_datetime(df['release_date'], errors='coerce') #errors='coerce' giúp chuyển những cái không phải ngày thành null
invalid_dates = parsed_dates.isnull().sum()
print(f"\nSố dòng có release_date không thể chuyển đổi thành ngày: {invalid_dates:,} dòng {invalid_dates / len(df) * 100:.2f}%")


# 3. Trích xuất năm phát hành sơ bộ để xem dải thời gian của dataset
years = parsed_dates.dt.year.dropna().astype(int)
print(f"\nNăm phát hành sớm nhất: {years.min()}")
print(f"Năm phát hành muộn nhất: {years.max()}")

--- KIỂM TRA NGÀY PHÁT HÀNH (RELEASE_DATE) ---
Số game đã phát hành chính thức: 58,041 (81.26%)
Số game chưa phát hành / Sắp ra mắt: 13,388 (18.74%)

Số dòng có release_date không thể chuyển đổi thành ngày: 71,401 dòng 99.96%

Năm phát hành sớm nhất: 2023
Năm phát hành muộn nhất: 2024


In [6]:
# Xem 15 mẫu giá trị thực tế khác nhau trong cột release_date
print("--- 15 MẪU GIÁ TRỊ THỰC TẾ CỦA RELEASE_DATE ---")
sample_dates = df['release_date'].dropna().drop_duplicates().head(15)
for i, val in enumerate(sample_dates, 1):
    print(f"{i:2d}. {repr(val)}")

# Thử nghiệm parse với tham số format='mixed' (hỗ trợ nhiều định dạng hỗn hợp)
parsed_mixed = pd.to_datetime(df['release_date'], format='mixed', errors='coerce')
valid_count = parsed_mixed.notnull().sum()
print(f"\nSố dòng parse thành công khi dùng format='mixed': {valid_count:,} / {len(df):,} ({valid_count/len(df)*100:.2f}%)")

if valid_count > 0:
    years_mixed = parsed_mixed.dt.year.dropna().astype(int)
    print(f"Dải năm phát hành: Từ {years_mixed.min()} đến {years_mixed.max()}")
    print("\n--- TOP 5 NĂM CÓ SỐ LƯỢNG GAME PHÁT HÀNH NHIỀU NHẤT ---")
    print(years_mixed.value_counts().head(5))

--- 15 MẪU GIÁ TRỊ THỰC TẾ CỦA RELEASE_DATE ---
 1. '2024-01-04'
 2. 'Not Released'
 3. '2024-01-26'
 4. '2024-02-17'
 5. '2024-01-05'
 6. '2024-01-15'
 7. '2024-03-26'
 8. '2024-02-29'
 9. '2024-05-23'
10. '2024-06-08'
11. '2024-04-06'
12. '2024-07-01'
13. '2024-01-08'
14. '2024-04-04'
15. '2024-01-09'

Số dòng parse thành công khi dùng format='mixed': 58,041 / 71,429 (81.26%)
Dải năm phát hành: Từ 1997 đến 2025

--- TOP 5 NĂM CÓ SỐ LƯỢNG GAME PHÁT HÀNH NHIỀU NHẤT ---
release_date
2024    9693
2023    7903
2021    7627
2019    6082
2020    5093
Name: count, dtype: int64


In [7]:
# 2. Xuất trực tiếp sang file Excel đúng 21 cột (từ A đến U)
excel_path = '../excel/steam_workbook.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Ghi dữ liệu vào sheet raw_data
    df.to_excel(writer, sheet_name='raw_data', index=False)

print(f"Đã xuất thành công {df.shape[0]:,} dòng và {df.shape[1]} cột sang {excel_path}!")

Đã xuất thành công 71,429 dòng và 21 cột sang ../excel/steam_workbook.xlsx!
